# Fine-tune EmbeddingGemma — Swiss Legal Retrieval
Dataset: `farwew/swiss-legal` (anchor, positive, negative triplets)

In [ ]:
!pip install -q -U sentence-transformers datasets
!pip install -q git+https://github.com/huggingface/transformers@v4.56.0-Embedding-Gemma-preview

In [ ]:
from huggingface_hub import login
login()  # masukkan HF token (butuh akses ke dataset private + model gemma)

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

model_id = 'google/embeddinggemma-300m'
model = SentenceTransformer(model_id, device=device)

# Gradient checkpointing — hemat ~60% VRAM, sedikit lebih lambat
model[0].auto_model.gradient_checkpointing_enable()
print('Gradient checkpointing enabled')
print('Max seq length:', model.max_seq_length)

In [ ]:
from datasets import load_dataset

ds = load_dataset('farwew/swiss-legal')
print(ds)
print('Columns:', ds['train'].column_names)
print('Train size:', len(ds['train']))

# Tampilkan sample
sample = ds['train'][0]
print('\nSample:')
print('  anchor  :', sample['anchor'][:100])
print('  positive:', sample['positive'][:100])
print('  negative:', sample['negative'][:100])

## Data Validation
Cek kualitas dataset sebelum training — null, duplikat, panjang teks, hard negative quality.

In [ ]:
import numpy as np

train_df = ds['train'].to_pandas()
split_name = 'eval' if 'eval' in ds else ('validation' if 'validation' in ds else None)
eval_df = ds[split_name].to_pandas() if split_name else None

print('=' * 55)
print('DATA VALIDATION REPORT')
print('=' * 55)

# 1. Required columns
required_cols = ['anchor', 'positive', 'negative']
missing = [c for c in required_cols if c not in train_df.columns]
assert not missing, f'Missing columns: {missing}'
print(f'[OK] Required columns: {required_cols}')

# 2. Null / empty check
for col in required_cols:
    n_null  = train_df[col].isna().sum()
    n_empty = (train_df[col].fillna('').str.strip() == '').sum()
    status = 'OK' if n_null == 0 and n_empty == 0 else 'WARN'
    print(f'[{status}] {col}: {n_null} null, {n_empty} empty')

# 3. Duplicate check
n_dup_rows  = train_df.duplicated(subset=required_cols).sum()
n_anc_eq_pos = (train_df['anchor'] == train_df['positive']).sum()
n_pos_eq_neg = (train_df['positive'] == train_df['negative']).sum()
print(f'[{"OK" if n_dup_rows == 0 else "WARN"}] Duplicate triplets: {n_dup_rows}')
print(f'[{"OK" if n_anc_eq_pos == 0 else "WARN"}] anchor == positive: {n_anc_eq_pos}')
print(f'[{"OK" if n_pos_eq_neg == 0 else "WARN"}] positive == negative: {n_pos_eq_neg}')

# 4. Text length distribution (chars)
print('\nText length (chars):')
for col in required_cols:
    lens = train_df[col].str.len()
    print(f'  {col:8s}: min={lens.min():<5} median={lens.median():<8.0f} max={lens.max():<6} mean={lens.mean():.0f}')

# 5. Approx token length (whitespace split)
print('\nApprox token count (whitespace split):')
for col in required_cols:
    toks = train_df[col].str.split().str.len()
    pct95 = int(np.percentile(toks, 95))
    print(f'  {col:8s}: median={toks.median():.0f}  95th={pct95}  max={toks.max()}  (model max={model.max_seq_length})')
    if pct95 > model.max_seq_length:
        print(f'  [WARN] >5% of {col} may be truncated at seq_len={model.max_seq_length}')

# 6. Eval split summary
if eval_df is not None:
    print(f'\nEval split ({split_name}): {len(eval_df)} rows, columns: {list(eval_df.columns)}')
    n_eval_null = eval_df[required_cols].isna().any(axis=1).sum()
    print(f'  Rows with any null: {n_eval_null}')
else:
    print('\n[WARN] No eval/validation split found — training without eval set')

# 7. Hard negative quality (cosine sim on 200 samples)
print('\nHard negative quality check (200 train samples):')
n_check = min(200, len(train_df))
sub = ds['train'].select(range(n_check))
a_emb = model.encode(sub['anchor'],   prompt_name='Retrieval-query',    normalize_embeddings=True, show_progress_bar=False)
p_emb = model.encode(sub['positive'], prompt_name='Retrieval-document', normalize_embeddings=True, show_progress_bar=False)
n_emb = model.encode(sub['negative'], prompt_name='Retrieval-document', normalize_embeddings=True, show_progress_bar=False)
pos_sim = (a_emb * p_emb).sum(axis=1)
neg_sim = (a_emb * n_emb).sum(axis=1)
margin  = pos_sim - neg_sim
n_correct = (pos_sim > neg_sim).sum()
print(f'  Positive > Negative : {n_correct}/{n_check} ({100*n_correct/n_check:.1f}%)')
print(f'  Avg positive sim    : {pos_sim.mean():.3f}')
print(f'  Avg negative sim    : {neg_sim.mean():.3f}')
print(f'  Avg margin (pos-neg): {margin.mean():.3f}')
if n_correct / n_check < 0.5:
    print('  [WARN] <50% positives rank above negatives — negatives may be mislabeled or too hard')
elif n_correct / n_check > 0.95:
    print('  [INFO] >95% correct — negatives are easy; consider mining harder negatives after initial training')
else:
    print('  [OK] Good difficulty range for training')

print('\n' + '=' * 55)
print('Validation complete. Proceed to training.')
print('=' * 55)

In [ ]:
# Cek similarity sebelum training
import numpy as np

samples = ds['train'].select(range(100))
a_emb = model.encode(samples['anchor'],   prompt_name='Retrieval-query',    normalize_embeddings=True)
p_emb = model.encode(samples['positive'], prompt_name='Retrieval-document', normalize_embeddings=True)
n_emb = model.encode(samples['negative'], prompt_name='Retrieval-document', normalize_embeddings=True)

pos_sim = (a_emb * p_emb).sum(axis=1)
neg_sim = (a_emb * n_emb).sum(axis=1)

print('=== SEBELUM TRAINING ===')
print(f'Positive > Negative: {(pos_sim > neg_sim).sum()}/100')
print(f'Avg positive sim   : {pos_sim.mean():.3f}')
print(f'Avg negative sim   : {neg_sim.mean():.3f}')

In [ ]:
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.losses import MultipleNegativesRankingLoss

train_dataset = ds['train'].select_columns(['anchor', 'positive', 'negative'])
eval_dataset  = ds['eval'].select_columns(['anchor', 'positive', 'negative']) if 'eval' in ds else None

loss = MultipleNegativesRankingLoss(model)

args = SentenceTransformerTrainingArguments(
    output_dir='finetuned-embeddinggemma-swiss-legal',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    learning_rate=1e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    lr_scheduler_type='cosine',
    fp16=torch.cuda.is_available(),
    prompts={
        'anchor':   model.prompts['Retrieval-query'],
        'positive': model.prompts['Retrieval-document'],
        'negative': model.prompts['Retrieval-document'],
    },
    logging_steps=50,
    report_to='none',
    save_strategy='epoch',
    save_total_limit=1,
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    loss=loss,
)
trainer.train()

In [ ]:
# Cek similarity setelah training
a_emb = model.encode(samples['anchor'],   prompt_name='Retrieval-query',    normalize_embeddings=True)
p_emb = model.encode(samples['positive'], prompt_name='Retrieval-document', normalize_embeddings=True)
n_emb = model.encode(samples['negative'], prompt_name='Retrieval-document', normalize_embeddings=True)

pos_sim = (a_emb * p_emb).sum(axis=1)
neg_sim = (a_emb * n_emb).sum(axis=1)

print('=== SETELAH TRAINING ===')
print(f'Positive > Negative: {(pos_sim > neg_sim).sum()}/100')
print(f'Avg positive sim   : {pos_sim.mean():.3f}')
print(f'Avg negative sim   : {neg_sim.mean():.3f}')

In [ ]:
# Simpan model
model.save('finetuned-embeddinggemma-swiss-legal/final')
print('Model saved.')

# Optional: push ke HuggingFace Hub
# model.push_to_hub('farwew/embeddinggemma-swiss-legal')